# Hyperparameter Tuning - Two Modern Approaches

**Razorpay AI Buildathon 2026 - Track 02: AI Risk Manager**

Return-Risk Scorer for COD orders (Return-to-Origin prediction).

---

### The comparison

Two search strategies, run head to head on the finalists from `03`:

1. **Optuna** — `TPESampler(multivariate=True)` + `HyperbandPruner`
2. **FLAML** — cost-frugal `BlendSearch` (CFO + global search)

### What is held identical

| Held identical | Why it matters |
|---|---|
| Search space, parameter for parameter | Declared once in `src/tuning.py` and translated into each dialect. A space that differs is a comparison of spaces, not of searchers. |
| Objective | Mean PR-AUC over the same expanding-window folds of **train** |
| Seed | `20260101` for both |
| **Wall-clock budget** | Optuna is trial-native, FLAML is budget-native. A fixed trial count would flatter Optuna, which spends whatever a trial costs. A fixed *time* budget is the fair meeting point — and trials completed becomes a **result**, not a control. |

### What is not touched

Validation is scored **once per tuned configuration**, after each search closes. It is
never part of the objective. The test set is not opened in this notebook.

### The question actually being asked

Not "which library is better" in the abstract, but: **given 5 minutes of the same CPU
on the same problem, which strategy hands back the better configuration — and is the
difference bigger than the difference between tuned and untuned at all?**

That last part matters. If tuning buys 0.002 PR-AUC over the `03` defaults, the honest
report is that this problem is not tuning-limited.

## 1. Load finalists & search space definition

In [ ]:
import json
import sys
import time
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.evaluate import classification_metrics
from src.features import TARGET
from src.models import SEED, build_pipeline, model_input_columns, model_zoo
from src.tuning import (
    apply_params,
    cheap_corner,
    optuna_history,
    run_flaml,
    run_optuna,
    space_for,
    space_summary,
)

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 110
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

FIG = ROOT / "reports/figures"
RES = ROOT / "reports/results"
MODELS = ROOT / "models"
MODELS.mkdir(exist_ok=True)

# The budget both strategies get, per finalist. Identical for both.
TIME_BUDGET_S = 300
N_TUNE_FOLDS = 3

In [ ]:
def load_split(name: str) -> pd.DataFrame:
    if name == "test":
        raise PermissionError(
            "04_hyperparameter_tuning may not read the test split. It is opened "
            "once, in 05_final_evaluation.ipynb, after the final model is selected."
        )
    return pd.read_parquet(ROOT / f"data/processed/{name}.parquet")


train = load_split("train")
val = load_split("val")
COLS = model_input_columns()
y_train = train[TARGET].to_numpy()
y_val = val[TARGET].to_numpy()

# Finalists come from 03's persisted output, not from a hardcoded list, so this
# notebook cannot silently disagree with the benchmark that chose them.
bench = json.load(open(RES / "03_benchmark_summary.json", encoding="utf-8"))
FINALISTS = bench["finalists"]
baseline = {r["key"]: r for r in bench["leaderboard"]}
CEILING = bench["bayes_ceiling_val"]

zoo = model_zoo(seed=SEED)
print("Finalists carried forward from 03:\n")
for k in FINALISTS:
    r = baseline[k]
    print(f"  {zoo[k].label:<24} untuned val PR-AUC {r['val_pr_auc']:.4f}   "
          f"CV {r['cv_pr_auc_mean']:.4f}")
print(f"\nBayes ceiling (val): PR-AUC {CEILING['pr_auc']:.4f}")
print(f"budget per (model, strategy): {TIME_BUDGET_S}s   |   objective folds: {N_TUNE_FOLDS}")

In [ ]:
# The space, shown once. Both libraries receive exactly this.
for k in FINALISTS:
    print(f"=== {zoo[k].label} ===")
    display(space_summary(k).style.hide(axis="index"))
    print(f"FLAML low-cost starting corner: {cheap_corner(k)}")
    print()

## 2. APPROACH 1 - Optuna: multivariate TPE + HyperbandPruner

In [ ]:
print("""Optuna configuration:

  sampler  TPESampler(multivariate=True, seed=SEED)
           multivariate models the JOINT distribution over parameters rather
           than each independently. For boosters that matters: learning rate and
           tree count trade off against each other, and an independent sampler
           cannot see that ridge.

  pruner   HyperbandPruner(min_resource=1, max_resource=3, reduction_factor=3)
           the objective reports an intermediate score after every CV fold, so a
           configuration that is clearly poor on fold 0 is stopped before it
           spends budget on folds 1 and 2. That is how Optuna gets more trials
           out of the same wall clock.
""")

OPTUNA_STUDIES, OPTUNA_HIST = {}, {}
for k in FINALISTS:
    t0 = time.time()
    study = run_optuna(zoo[k], train, y_train, COLS,
                       time_budget_s=TIME_BUDGET_S, n_splits=N_TUNE_FOLDS, seed=SEED)
    hist = optuna_history(study)
    OPTUNA_STUDIES[k], OPTUNA_HIST[k] = study, hist

    done = (hist.state == "COMPLETE").sum()
    pruned = (hist.state == "PRUNED").sum()
    print(f"{zoo[k].label:<24} best CV PR-AUC {study.best_value:.4f}   "
          f"{done} complete + {pruned} pruned in {time.time() - t0:.0f}s")

## 3. Optuna study analysis (importance, parallel coordinate, slice)

In [ ]:
import optuna

imp_rows = []
for k in FINALISTS:
    try:
        imp = optuna.importance.get_param_importances(OPTUNA_STUDIES[k])
    except Exception as exc:
        print(f"{zoo[k].label}: importance unavailable ({exc})")
        continue
    for p, v in imp.items():
        imp_rows.append({"model": zoo[k].label, "parameter": p, "importance": v})

importance = pd.DataFrame(imp_rows)
if len(importance):
    fig, axes = plt.subplots(1, len(FINALISTS), figsize=(5.2 * len(FINALISTS), 4))
    axes = np.atleast_1d(axes)
    for ax, k in zip(axes, FINALISTS):
        sub = (importance[importance.model == zoo[k].label]
               .sort_values("importance"))
        ax.barh(sub.parameter, sub.importance, color="#68a")
        ax.set_title(f"{zoo[k].label}\nparameter importance", fontsize=10)
        ax.set_xlabel("share of objective variance explained")
    plt.tight_layout()
    plt.savefig(FIG / "04_optuna_param_importance.png", bbox_inches="tight")
    plt.show()
    display(importance.pivot(index="parameter", columns="model",
                             values="importance").round(3))

In [ ]:
# Slice plots: what the sampler learned about each parameter, one at a time.
lead = FINALISTS[0]
study = OPTUNA_STUDIES[lead]
params = [p.name for p in space_for(lead) if p.kind != "cat"]
params = [p for p in params if p in study.best_params]

ncol = min(4, len(params))
nrow = int(np.ceil(len(params) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(4 * ncol, 3.2 * nrow), squeeze=False)
completed = [t for t in study.trials if t.value is not None]
for ax, p in zip(axes.ravel(), params):
    xs = [t.params[p] for t in completed if p in t.params]
    ys = [t.value for t in completed if p in t.params]
    ax.scatter(xs, ys, s=22, c=range(len(xs)), cmap="viridis")
    ax.axvline(study.best_params[p], ls="--", c="#c44", lw=1.4)
    ax.set_xlabel(p)
    ax.set_ylabel("CV PR-AUC")
    spec = next(s for s in space_for(lead) if s.name == p)
    if spec.log:
        ax.set_xscale("log")
for ax in axes.ravel()[len(params):]:
    ax.axis("off")
fig.suptitle(f"Optuna slice plots - {zoo[lead].label} "
             "(colour = trial order, dashed = best)", y=1.01)
plt.tight_layout()
plt.savefig(FIG / "04_optuna_slice.png", bbox_inches="tight")
plt.show()

In [ ]:
# Parallel coordinates over the top quartile of trials: where the good region is.
fig, ax = plt.subplots(figsize=(12, 4.6))
comp = pd.DataFrame([
    {**t.params, "value": t.value}
    for t in OPTUNA_STUDIES[lead].trials if t.value is not None
])
numeric = [c for c in comp.columns
           if c != "value" and pd.api.types.is_numeric_dtype(comp[c])]
norm = comp[numeric].apply(
    lambda s: (s - s.min()) / (s.max() - s.min()) if s.max() > s.min() else s * 0)
cut = comp.value.quantile(0.75)
for i, row in norm.iterrows():
    good = comp.value.iat[i] >= cut
    ax.plot(range(len(numeric)), row.values,
            color="#c44" if good else "#ccc",
            alpha=0.85 if good else 0.35, lw=1.6 if good else 0.8, zorder=3 if good else 1)
ax.set_xticks(range(len(numeric)), numeric, rotation=25, ha="right")
ax.set_ylabel("normalised parameter value")
ax.set_title(f"Parallel coordinates - {zoo[lead].label} "
             "(red = top-quartile trials by CV PR-AUC)")
plt.tight_layout()
plt.savefig(FIG / "04_optuna_parallel_coords.png", bbox_inches="tight")
plt.show()

## 4. APPROACH 2 - FLAML: cost-frugal BlendSearch/CFO

In [ ]:
print("""FLAML configuration:

  searcher  BlendSearch(seed=SEED, low_cost_partial_config=<cheap corner>)
            BlendSearch runs CFO (a local, cost-aware search) alongside a global
            search and blends them.

  The premise: evaluation cost varies enormously across the space. 900 trees cost
  roughly nine times what 100 do, so a searcher that ignores cost spends most of
  its budget in the expensive corner. BlendSearch starts from a CHEAP
  configuration and expands outward, buying many cheap trials instead of a few
  expensive ones.

  low_cost_partial_config is where that prior is declared. Leaving it empty would
  throw away the entire point of the method, so it is specified per model in
  src/tuning.py::cheap_corner and printed above -- visible and arguable rather
  than hidden.

  Same space. Same seed. Same objective. Same {}s.
""".format(TIME_BUDGET_S))

FLAML_HIST, FLAML_BEST = {}, {}
for k in FINALISTS:
    t0 = time.time()
    analysis, hist = run_flaml(zoo[k], train, y_train, COLS,
                               time_budget_s=TIME_BUDGET_S, n_splits=N_TUNE_FOLDS,
                               seed=SEED, low_cost_hint=cheap_corner(k))
    FLAML_HIST[k] = hist
    best_row = hist.loc[hist.value.idxmax()]
    FLAML_BEST[k] = {"value": float(best_row.value), "config": best_row.config}
    print(f"{zoo[k].label:<24} best CV PR-AUC {best_row.value:.4f}   "
          f"{len(hist)} trials in {time.time() - t0:.0f}s")

## 5. Head-to-head: same budget, same space, same seed

In [ ]:
rows = []
for k in FINALISTS:
    oh, fh = OPTUNA_HIST[k], FLAML_HIST[k]
    o_best = OPTUNA_STUDIES[k].best_value
    f_best = FLAML_BEST[k]["value"]
    rows.append({
        "model": zoo[k].label,
        "optuna_best_cv": o_best,
        "flaml_best_cv": f_best,
        "delta_optuna_minus_flaml": o_best - f_best,
        "optuna_trials": int((oh.state == "COMPLETE").sum()),
        "optuna_pruned": int((oh.state == "PRUNED").sum()),
        "flaml_trials": len(fh),
        "untuned_cv": baseline[k]["cv_pr_auc_mean"],
    })
h2h = pd.DataFrame(rows)
h2h["winner"] = np.where(h2h.delta_optuna_minus_flaml > 0, "Optuna", "FLAML")
h2h["tuning_gain_vs_untuned"] = (
    h2h[["optuna_best_cv", "flaml_best_cv"]].max(axis=1) - h2h.untuned_cv)
display(h2h.round(4).style.hide(axis="index"))

In [ ]:
print(f"""Read the trial counts before the scores.

Both strategies had exactly {TIME_BUDGET_S} seconds. The number of configurations each
got through is a RESULT of its cost model, not a setting we chose:

""")
for _, r in h2h.iterrows():
    tot_o = r.optuna_trials + r.optuna_pruned
    print(f"  {r.model:<24} Optuna {tot_o:>3} trials "
          f"({r.optuna_pruned} pruned early)   FLAML {r.flaml_trials:>3} trials")
print(f"""
Optuna's Hyperband pruner and FLAML's cost-frugal search are two different
answers to the same problem -- that a fixed budget buys either a few careful
evaluations or many cheap ones. Neither is universally right; this table is what
happened on THIS problem at THIS budget.""")

## 6. Convergence comparison

In [ ]:
fig, axes = plt.subplots(1, len(FINALISTS), figsize=(5.6 * len(FINALISTS), 4.2),
                         squeeze=False)
for ax, k in zip(axes[0], FINALISTS):
    oh = OPTUNA_HIST[k].dropna(subset=["elapsed_s"])
    fh = FLAML_HIST[k]
    ax.step(oh.elapsed_s, oh.running_best, where="post", lw=2,
            label=f"Optuna (TPE+Hyperband)", color="#c44")
    ax.step(fh.elapsed_s, fh.running_best, where="post", lw=2,
            label="FLAML (BlendSearch)", color="#4a8")
    ax.axhline(baseline[k]["cv_pr_auc_mean"], ls="--", c="k", lw=1.2,
               label="untuned default (03)")
    ax.set_xlabel("elapsed seconds (the shared budget axis)")
    ax.set_ylabel("best CV PR-AUC so far")
    ax.set_title(zoo[k].label)
    ax.legend(fontsize=8, loc="lower right")
plt.tight_layout()
plt.savefig(FIG / "04_convergence.png", bbox_inches="tight")
plt.show()

print("""Time is the honest x-axis here, not trial number. Plotting against trial
index would show FLAML "converging slower" purely because it takes more, cheaper
steps -- which is the strategy working as designed, not failing.""")

In [ ]:
# How fast did each strategy reach within 0.001 PR-AUC of its own final answer?
speed = []
for k in FINALISTS:
    for name, hist, best in [
        ("Optuna", OPTUNA_HIST[k].dropna(subset=["elapsed_s"]), OPTUNA_STUDIES[k].best_value),
        ("FLAML", FLAML_HIST[k], FLAML_BEST[k]["value"]),
    ]:
        reached = hist[hist.running_best >= best - 0.001]
        speed.append({
            "model": zoo[k].label, "strategy": name,
            "final_cv_pr_auc": best,
            "seconds_to_within_0.001": (reached.elapsed_s.iloc[0]
                                        if len(reached) else np.nan),
            "trials_to_get_there": (int(reached.trial.iloc[0]) + 1
                                    if len(reached) else np.nan),
        })
display(pd.DataFrame(speed).round(4).style.hide(axis="index"))

## 7. Best configuration per approach

In [ ]:
configs = []
for k in FINALISTS:
    configs.append({"model": zoo[k].label, "strategy": "Optuna",
                    "cv_pr_auc": OPTUNA_STUDIES[k].best_value,
                    "config": OPTUNA_STUDIES[k].best_params})
    configs.append({"model": zoo[k].label, "strategy": "FLAML",
                    "cv_pr_auc": FLAML_BEST[k]["value"],
                    "config": FLAML_BEST[k]["config"]})

for c in configs:
    print(f"{c['model']} / {c['strategy']}  ->  CV PR-AUC {c['cv_pr_auc']:.4f}")
    for p, v in sorted(c["config"].items()):
        print(f"    {p:<22} {v}")
    print()

In [ ]:
# Did the two strategies land in the same region, or find different optima?
lead_cfgs = {c["strategy"]: c["config"] for c in configs if c["model"] == zoo[lead].label}
common = sorted(set(lead_cfgs["Optuna"]) & set(lead_cfgs["FLAML"]))
comp_rows = []
for p in common:
    o, f = lead_cfgs["Optuna"][p], lead_cfgs["FLAML"][p]
    spec = next((s for s in space_for(lead) if s.name == p), None)
    if spec and spec.kind != "cat" and spec.high is not None:
        span = spec.high - spec.low
        gap = abs(float(o) - float(f)) / span if span else 0.0
    else:
        gap = np.nan
    comp_rows.append({"parameter": p, "optuna": o, "flaml": f,
                      "gap_as_share_of_range": gap})
display(pd.DataFrame(comp_rows).round(4).style.hide(axis="index"))

## 8. Refit on train, score on val

In [ ]:
# Validation is scored ONCE per tuned configuration, now that both searches
# have closed. It was never part of the objective.
from sklearn.pipeline import Pipeline
from src.models import build_preprocessor

tuned_rows, TUNED_PIPES, TUNED_VAL_PRED = [], {}, {}

for k in FINALISTS:
    for strategy, params in [("Optuna", OPTUNA_STUDIES[k].best_params),
                             ("FLAML", FLAML_BEST[k]["config"])]:
        pipe = Pipeline([
            ("pre", build_preprocessor(scale=zoo[k].scale)),
            ("clf", apply_params(zoo[k], dict(params))),
        ])
        t0 = time.time()
        pipe.fit(train[COLS], y_train)
        p_val = pipe.predict_proba(val[COLS])[:, 1]
        m = classification_metrics(y_val, p_val)
        tag = f"{k}__{strategy.lower()}"
        TUNED_PIPES[tag], TUNED_VAL_PRED[tag] = pipe, p_val
        tuned_rows.append({
            "key": tag, "model": zoo[k].label, "strategy": strategy,
            "val_pr_auc": m["pr_auc"], "val_roc_auc": m["roc_auc"],
            "val_brier": m["brier"], "val_log_loss": m["log_loss"],
            "untuned_val_pr_auc": baseline[k]["val_pr_auc"],
            "gain_vs_untuned": m["pr_auc"] - baseline[k]["val_pr_auc"],
            "fit_seconds": time.time() - t0,
        })

tuned = pd.DataFrame(tuned_rows).sort_values("val_pr_auc", ascending=False)
display(tuned.round(4).style.background_gradient(
    subset=["val_pr_auc"], cmap="Greens").hide(axis="index"))

assert tuned.val_pr_auc.max() <= CEILING["pr_auc"] + 1e-9, "tuned model beat the ceiling"
print("\nceiling check passed")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.6))

w = 0.35
models = [zoo[k].label for k in FINALISTS]
x = np.arange(len(models))
unt = [baseline[k]["val_pr_auc"] for k in FINALISTS]
opt = [tuned[(tuned.model == zoo[k].label) & (tuned.strategy == "Optuna")].val_pr_auc.iat[0]
       for k in FINALISTS]
fla = [tuned[(tuned.model == zoo[k].label) & (tuned.strategy == "FLAML")].val_pr_auc.iat[0]
       for k in FINALISTS]
ax[0].bar(x - w, unt, w, label="untuned (03)", color="#999")
ax[0].bar(x, opt, w, label="Optuna", color="#c44")
ax[0].bar(x + w, fla, w, label="FLAML", color="#4a8")
ax[0].axhline(CEILING["pr_auc"], ls="--", c="k", lw=1.4, label="Bayes ceiling")
ax[0].set_xticks(x, models, rotation=12)
ax[0].set_ylabel("validation PR-AUC")
ax[0].set_title("Did tuning buy anything?")
ax[0].legend(fontsize=8)
lo = min(unt + opt + fla)
ax[0].set_ylim(lo - 0.02, CEILING["pr_auc"] + 0.01)

gains = tuned.groupby("strategy").gain_vs_untuned
ax[1].axhline(0, c="k", lw=1)
for i, (s, g) in enumerate(gains):
    ax[1].bar(np.arange(len(g)) + i * 0.35, g.values, 0.35,
              label=s, color="#c44" if s == "Optuna" else "#4a8")
ax[1].set_xticks(np.arange(len(FINALISTS)) + 0.18, models, rotation=12)
ax[1].set_ylabel("val PR-AUC gain over the 03 default")
ax[1].set_title("Tuning gain, per model per strategy")
ax[1].legend(fontsize=8)
plt.tight_layout()
plt.savefig(FIG / "04_tuned_vs_untuned.png", bbox_inches="tight")
plt.show()

In [ ]:
best_tuned = tuned.iloc[0]
best_untuned_val = max(baseline[k]["val_pr_auc"] for k in FINALISTS)
overall_gain = best_tuned.val_pr_auc - best_untuned_val

print(f"""HONEST READ ON TUNING

  best untuned finalist (03) : {best_untuned_val:.4f} val PR-AUC
  best tuned configuration   : {best_tuned.val_pr_auc:.4f}  ({best_tuned.model} / {best_tuned.strategy})
  net gain from tuning       : {overall_gain:+.4f}
  Bayes ceiling              : {CEILING['pr_auc']:.4f}
  remaining headroom         : {CEILING['pr_auc'] - best_tuned.val_pr_auc:.4f}
""")

if overall_gain < 0.005:
    print("""  A gain this small is not a tuning success story, and we are not going to
  dress it up as one. The problem is not hyperparameter-limited: 01 built the
  labels from an additive log-odds model with substantial Bernoulli noise, so the
  achievable ceiling is close and every reasonable configuration sits near it.

  What the exercise still bought:
    - evidence that the 03 defaults were not accidentally poor
    - a defensible comparison of two search strategies under a fixed budget
    - parameter-importance evidence about which knobs matter at all

  Where tuning WOULD pay on real merchant data: interactions between courier,
  corridor, seller and season that a default-depth booster underfits.""")
else:
    print(f"  Tuning moved validation PR-AUC by {overall_gain:+.4f}, which is a real gain.")

## 9. Persist tuned models

In [ ]:
artifacts = []
for tag, pipe in TUNED_PIPES.items():
    p = MODELS / f"04_{tag}.joblib"
    joblib.dump(pipe, p)
    artifacts.append(p)

pred = pd.DataFrame({"order_id": val.order_id.to_numpy(), "y_true": y_val})
for tag, pv in TUNED_VAL_PRED.items():
    pred[tag] = pv
pred.to_parquet(RES / "04_tuned_val_predictions.parquet", index=False)

hist_out = pd.concat(
    [OPTUNA_HIST[k].assign(model=zoo[k].label, strategy="Optuna") for k in FINALISTS]
    + [FLAML_HIST[k].drop(columns=["config"]).assign(model=zoo[k].label, strategy="FLAML")
       for k in FINALISTS],
    ignore_index=True)
hist_out.to_csv(RES / "04_search_history.csv", index=False)

with open(RES / "04_tuning_summary.json", "w", encoding="utf-8") as fh:
    json.dump({
        "seed": SEED,
        "time_budget_s_per_model_per_strategy": TIME_BUDGET_S,
        "objective": {"metric": "pr_auc", "folds": N_TUNE_FOLDS,
                      "scheme": "expanding-window TimeSeriesSplit on train"},
        "finalists": FINALISTS,
        "head_to_head": h2h.to_dict(orient="records"),
        "search_speed": speed,
        "best_configs": [{**c, "config": {k2: (list(v) if isinstance(v, tuple) else v)
                                          for k2, v in c["config"].items()}}
                         for c in configs],
        "tuned_val_scores": tuned.to_dict(orient="records"),
        "param_importance": (importance.to_dict(orient="records")
                             if len(importance) else []),
        "bayes_ceiling_val": CEILING,
        "net_gain_vs_untuned": float(overall_gain),
    }, fh, indent=2, default=str)

print(f"persisted {len(artifacts)} tuned pipelines to models/")
for p in artifacts:
    print(f"  {p.relative_to(ROOT)}")
for f in ["04_tuned_val_predictions.parquet", "04_search_history.csv",
          "04_tuning_summary.json"]:
    print(f"wrote reports/results/{f}")

In [ ]:
wins = h2h.winner.value_counts()
print(f"""SUMMARY

  budget                {TIME_BUDGET_S}s per model per strategy, identical space and seed
  finalists tuned       {', '.join(zoo[k].label for k in FINALISTS)}
  strategy wins (CV)    {dict(wins)}
  best tuned            {best_tuned.model} / {best_tuned.strategy} at {best_tuned.val_pr_auc:.4f} val PR-AUC
  net gain over 03      {overall_gain:+.4f}
  Bayes ceiling         {CEILING['pr_auc']:.4f}

  test set              NOT OPENED

  Model selection happens in 05 on VALIDATION ONLY, and then the test set is
  opened once.""")

---

## What 04 established

| | |
|---|---|
| Comparison | Optuna (multivariate TPE + Hyperband) vs FLAML (BlendSearch/CFO) |
| Held identical | space, objective, folds, seed, wall-clock budget |
| Trials completed | a *result* of each cost model, not a control |
| Convergence axis | elapsed seconds — trial index would flatter the expensive searcher |
| Validation | scored once per tuned config, after both searches closed |

**Next:** `05_final_evaluation.ipynb` — select one model on validation, open the test
set once, and report cost, calibration, the three-tier policy, the prevalence-shift
study, SHAP reasons and the honest exception list.